[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module5/03-regression.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module5/03-regression.ipynb)

# Module 5 — Lesson 3: Regression

**Module:** 5 — Machine Learning Foundations | **Time:** 45 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Fit and interpret `LinearRegression` models
- Apply `PolynomialFeatures` to capture non-linear relationships
- Regularise models with `Ridge`, `Lasso`, and `ElasticNet`
- Evaluate regression models using MSE, RMSE, MAE, R², and adjusted R²
- Produce and interpret residual plots and predicted-vs-actual plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Libraries loaded.')

## 1. Dataset — California Housing

We use the California Housing dataset (20 640 samples, 8 features) to predict median house value. The `fetch_california_housing` function downloads it from the web if needed — no local files required.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print('Shape:', df.shape)
print('\nFeature descriptions:')
for name, desc in zip(housing.feature_names, housing.DESCR.split('\n')[25:33]):
    print(f'  {name:12s}: {desc.strip()}')
print('\nTarget: MedHouseVal (median house value in $100k)')
print(df.describe().round(2))

## 2. Linear Regression

Linear regression models the target as a weighted sum of features:

```
y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ + ε
```

Coefficients are found by minimising the sum of squared residuals (Ordinary Least Squares).

In [ ]:
X = df.drop('MedHouseVal', axis=1).values
y = df['MedHouseVal'].values
feature_names = housing.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])
pipe_lr.fit(X_train, y_train)
y_pred = pipe_lr.predict(X_test)

coefs = pipe_lr.named_steps['reg'].coef_
intercept = pipe_lr.named_steps['reg'].intercept_

print('Linear Regression Coefficients (on standardised features):')
for name, coef in zip(feature_names, coefs):
    print(f'  {name:12s}: {coef:+.4f}')
print(f'  intercept   : {intercept:.4f}')

## 3. Regression Metrics

| Metric | Formula | Notes |
|---|---|---|
| **MSE** | mean((y - ŷ)²) | Penalises large errors heavily |
| **RMSE** | √MSE | Same units as target |
| **MAE** | mean(|y - ŷ|) | More robust to outliers |
| **R²** | 1 - SS_res/SS_tot | 1 = perfect, 0 = baseline mean model |
| **Adj. R²** | 1 - (1-R²)(n-1)/(n-p-1) | Penalises adding uninformative features |

In [ ]:
def regression_metrics(y_true, y_pred, n_features, label='Model'):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    n    = len(y_true)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    print(f'--- {label} ---')
    print(f'  MSE:     {mse:.4f}')
    print(f'  RMSE:    {rmse:.4f}  (in $100k units)')
    print(f'  MAE:     {mae:.4f}')
    print(f'  R²:      {r2:.4f}')
    print(f'  Adj R²:  {adj_r2:.4f}')
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'AdjR2': adj_r2}

results = {}
results['LinearRegression'] = regression_metrics(y_test, y_pred, X.shape[1], 'Linear Regression')

## 4. Residual Plot and Predicted vs Actual

A good regression model should have residuals that:
- Are centred around zero (no systematic bias)
- Have roughly constant variance across the range of predictions (homoscedasticity)
- Show no obvious pattern

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred, alpha=0.3, s=8, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()

# Residuals vs Predicted
axes[1].scatter(y_pred, residuals, alpha=0.3, s=8, color='darkorange')
axes[1].axhline(0, color='red', lw=2, linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Plot')

plt.suptitle('Linear Regression Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Residual mean: {residuals.mean():.4f}  |  std: {residuals.std():.4f}')

## 5. Polynomial Regression

`PolynomialFeatures` expands the feature space with interaction terms and higher-degree terms, enabling linear models to fit non-linear relationships.

In [ ]:
# Use a single feature for clarity of visualisation
X_single = X[:, 0].reshape(-1, 1)  # MedInc
y_target  = y

X_tr1, X_te1, y_tr1, y_te1 = train_test_split(X_single, y_target, test_size=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
X_plot_range = np.linspace(X_single.min(), X_single.max(), 300).reshape(-1, 1)

for ax, deg in zip(axes, [1, 2, 3]):
    pipe_poly = Pipeline([
        ('poly',   PolynomialFeatures(degree=deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg',    LinearRegression())
    ])
    pipe_poly.fit(X_tr1, y_tr1)
    y_plot = pipe_poly.predict(X_plot_range)
    r2_te  = r2_score(y_te1, pipe_poly.predict(X_te1))

    ax.scatter(X_te1, y_te1, alpha=0.15, s=5, color='steelblue')
    ax.plot(X_plot_range, y_plot, color='tomato', lw=2)
    ax.set_title(f'Degree {deg}  |  Test R²={r2_te:.3f}')
    ax.set_xlabel('Median Income')
    ax.set_ylabel('House Value')

plt.suptitle('Polynomial Regression Fits (MedInc → HouseValue)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Ridge and Lasso Regularisation

Regularisation adds a penalty on large coefficients to the loss function:

- **Ridge (L2):** adds `α * Σβᵢ²` — shrinks all coefficients toward zero, never exactly zero
- **Lasso (L1):** adds `α * Σ|βᵢ|` — can zero out coefficients entirely (feature selection)
- **ElasticNet:** combines L1 and L2 penalties

The hyperparameter `α` controls the strength of regularisation.

In [ ]:
alphas = np.logspace(-3, 3, 50)

ridge_r2s, lasso_r2s = [], []

for alpha in alphas:
    pipe_r = Pipeline([('sc', StandardScaler()), ('reg', Ridge(alpha=alpha))])
    pipe_l = Pipeline([('sc', StandardScaler()), ('reg', Lasso(alpha=alpha, max_iter=5000))])
    ridge_r2s.append(cross_val_score(pipe_r, X_train, y_train, cv=3, scoring='r2').mean())
    lasso_r2s.append(cross_val_score(pipe_l, X_train, y_train, cv=3, scoring='r2').mean())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, scores, name, color in zip(axes, [ridge_r2s, lasso_r2s], ['Ridge', 'Lasso'], ['steelblue', 'tomato']):
    ax.semilogx(alphas, scores, color=color, lw=2)
    best_a = alphas[np.argmax(scores)]
    ax.axvline(best_a, linestyle='--', color='black', label=f'Best α={best_a:.3f}')
    ax.set_xlabel('Alpha (regularisation strength)')
    ax.set_ylabel('Mean CV R²')
    ax.set_title(f'{name} — CV R² vs Alpha')
    ax.legend()

plt.suptitle('Ridge and Lasso Alpha Tuning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_ridge_alpha = alphas[np.argmax(ridge_r2s)]
best_lasso_alpha = alphas[np.argmax(lasso_r2s)]
print(f'Best Ridge alpha: {best_ridge_alpha:.4f}  |  CV R²: {max(ridge_r2s):.4f}')
print(f'Best Lasso alpha: {best_lasso_alpha:.4f}  |  CV R²: {max(lasso_r2s):.4f}')

## 7. Lasso Feature Selection (Sparse Coefficients)

In [ ]:
# Fit Lasso with the best alpha and inspect which coefficients are zeroed
pipe_lasso_best = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Lasso(alpha=best_lasso_alpha, max_iter=5000))
])
pipe_lasso_best.fit(X_train, y_train)
lasso_coefs = pipe_lasso_best.named_steps['reg'].coef_

# Compare with Ridge
pipe_ridge_best = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=best_ridge_alpha))
])
pipe_ridge_best.fit(X_train, y_train)
ridge_coefs = pipe_ridge_best.named_steps['reg'].coef_

x_pos = np.arange(len(feature_names))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, coefs, name, color in zip(axes, [ridge_coefs, lasso_coefs],
                                   ['Ridge', 'Lasso'], ['steelblue', 'tomato']):
    ax.bar(x_pos, coefs, color=color, alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(feature_names, rotation=35, ha='right')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Coefficient')
    ax.set_title(f'{name} Coefficients (standardised)')
    non_zero = np.sum(coefs != 0)
    ax.set_xlabel(f'{non_zero}/{len(coefs)} non-zero coefficients')

plt.suptitle('Ridge vs Lasso — Coefficient Sparsity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Final metrics comparison
models = {
    'LinearRegression': (pipe_lr, X_test, y_test),
    'Ridge':            (pipe_ridge_best, X_test, y_test),
    'Lasso':            (pipe_lasso_best, X_test, y_test),
}
print('\n--- Final Test Metrics ---')
for name, (pipe, Xt, yt) in models.items():
    yp = pipe.predict(Xt)
    print(f'{name:20s} | R²={r2_score(yt,yp):.4f} | RMSE={np.sqrt(mean_squared_error(yt,yp)):.4f}')

## 8. ElasticNet

`ElasticNet` combines both Ridge and Lasso penalties. The `l1_ratio` parameter controls the balance: 0 = pure Ridge, 1 = pure Lasso.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe_en = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', ElasticNet(max_iter=5000))
])

param_grid = {
    'reg__alpha':    [0.01, 0.1, 1.0],
    'reg__l1_ratio': [0.1, 0.5, 0.9]
}

grid = GridSearchCV(pipe_en, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

print('ElasticNet Grid Search Results:')
print('Best params:  ', grid.best_params_)
print('Best CV R²:   ', round(grid.best_score_, 4))
print('Test R²:      ', round(r2_score(y_test, grid.predict(X_test)), 4))
print('Test RMSE:    ', round(np.sqrt(mean_squared_error(y_test, grid.predict(X_test))), 4))

## Practice Exercises

**Exercise 1 — Coefficient Interpretation**
Using the fitted `LinearRegression` pipeline from section 2, identify the three features with the largest positive coefficients and the three with the largest negative coefficients. Write a comment interpreting what each top-3 feature means economically (e.g., higher median income → higher house value).

**Exercise 2 — Polynomial + Regularisation**
Build a pipeline: `PolynomialFeatures(degree=2)` → `StandardScaler` → `Ridge`. Use `GridSearchCV` over `poly__degree=[1,2,3]` and `ridge__alpha=[0.01, 0.1, 1.0, 10.0]`. Report the best combination and its test RMSE.

**Exercise 3 — Residual Normality Check**
Fit the best model from Exercise 2. Plot a histogram of its residuals on the test set. Overlay a normal distribution curve using `scipy.stats.norm`. Does the residual distribution look approximately normal? What does non-normality suggest about the model?